In [ ]:
def min_max_normalize(data, range_min=0, range_max=1):
    """
    对数据进行最大最小归一化

    参数:
    data (numpy.ndarray): 输入数据数组
    range_min (float): 归一化后的最小值
    range_max (float): 归一化后的最大值

    返回:
    numpy.ndarray: 归一化后的数据
    """
    
    data_min = np.min(data)
    data_max = np.max(data)
    # 避免除以零
    if data_max - data_min == 0:
        return np.full_like(data, range_min)
    normalized_data = (data - data_min) / (data_max - data_min)  # 缩放到 [0, 1]
    scaled_data = normalized_data * (range_max - range_min) + range_min  # 缩放到 [range_min, range_max]
    return scaled_data


def mean_angular_deviation(vectors):
    # 将向量转换为单位向量
    unit_vectors = [v / np.linalg.norm(v) for v in vectors]

    # 计算平均单位向量
    mean_vector = np.mean(unit_vectors, axis=0)
    mean_vector /= np.linalg.norm(mean_vector)  # 单位化平均向量

    # 计算每个单位向量与平均单位向量之间的夹角
    angles = []
    for v in unit_vectors:
        # 计算夹角（反余弦）
        cos_theta = np.dot(v, mean_vector)
        # 限制 cos_theta 的范围 [-1, 1]，避免数值误差
        cos_theta = np.clip(cos_theta, -1.0, 1.0)
        angle = np.arccos(cos_theta)  # 弧度
        angles.append(angle)

    # 计算均值偏差（所有夹角的平均值）
    mad = np.mean(angles)
    return mad


import numpy as np


def direction_dispersion(vectors):
    # 将每个向量转换为单位向量
    unit_vectors = np.array([v / np.linalg.norm(v) for v in vectors])

    # 计算均值单位向量
    mean_vector = np.mean(unit_vectors, axis=0)
    mean_vector /= np.linalg.norm(mean_vector)  # 单位化均值向量

    # 计算每个向量与均值向量之间的夹角（反余弦）
    angles = []
    for v in unit_vectors:
        cos_theta = np.dot(v, mean_vector)
        cos_theta = np.clip(cos_theta, -1.0, 1.0)  # 限制在[-1, 1]之间
        angle = np.arccos(cos_theta)
        angles.append(angle)

    # 计算夹角的标准差
    angle_std = np.std(angles)

    # 返回标准差作为散度的度量
    return angle_std


# 示例：一组三维向量
vectors = np.array([
    [1, 0, 0],
    [0.9, 0.4, 0],
    [1, 0, 0.1],
    [0.8, 0.5, 0]
])

dispersion = direction_dispersion(vectors)
print(f"Direction Dispersion (Standard Deviation of Angles): {dispersion}")


def compute_vectors_from_first_point(points):
    # 第一个点
    first_point = np.array(points[0])

    # 计算每个点与第一个点的差向量
    vectors = [np.array(p) - first_point for p in points[1:]]

    return vectors


# data = X
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from itertools import combinations
import math
from tqdm import tqdm
from matplotlib import pyplot as plt
from scipy.spatial import KDTree

data = pd.read_csv("./datasets/aggregation.csv")
data = np.array(data)

In [ ]:
# 假设 'data' 是一个包含二维数据点的列表或数组
# np.random.shuffle(data)
x_values = data[:, 0]
y_values = data[:, 1]
# y_ = data[:, 2]
points = np.array(data)
n_points = len(points)

# 构建KDTree
tree = KDTree(points)
plt.scatter(x_values, y_values,s=5)
# plt.scatter(x_values, y_values,s=5)
print(len(data))

In [ ]:
see_pnum = 25
# see_pnum = int(len(data) * 0.02)

print(see_pnum)
no_flag_data = data[:, :2]
variances = []
means = []
r_s = []
mads = []
for p in data:
    distances, indices = tree.query(p, k=see_pnum)
    distances = distances[1:]
    variance = np.var(distances)
    mean = np.mean(distances)
    variances.append(variance)
    means.append(mean)
    r_s.append(np.median(distances))
    # 计算 向量组
    pp = no_flag_data[indices] #周围的点
    re = compute_vectors_from_first_point(pp)
    mad = direction_dispersion(re)
    mads.append(mad)

variances_1 = [1 / e for e in variances]
variances_1 = np.array(variances_1)
means_1 = [1/ e for e in means]
means_1 = np.array(means_1)
r_s = np.array(r_s)  # 的到每个点的半径
r_s_1 =np.array([1/e for e in r_s]) 
nm_variances_1 = min_max_normalize(variances_1,0,1)
nm_r_s_1 = min_max_normalize(r_s_1,0,1)
gsdata = np.column_stack((nm_variances_1,nm_r_s_1))

In [ ]:
variances_1

In [ ]:
Hadamard = min_max_normalize(variances_1,0,1) * min_max_normalize(r_s_1,0,1) 
maxh = np.argmax(Hadamard)
minh = np.argmin(Hadamard)
initial_means = np.array([gsdata[maxh], gsdata[minh]])
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.mixture import GaussianMixture

# 示例数据

# GMM 拟合
gmm = GaussianMixture(n_components=2, covariance_type='full')
gmm.fit(gsdata)

# 提取参数
means = gmm.means_
covariances = gmm.covariances_
weights = gmm.weights_

# 网格点
x = np.linspace(gsdata[:, 0].min() - 1, gsdata[:, 0].max() + 1, 500)
y = np.linspace(gsdata[:, 1].min() - 1, gsdata[:, 1].max() + 1, 500)
X, Y = np.meshgrid(x, y)
xy = np.vstack([X.ravel(), Y.ravel()]).T

# 绘制数据点
plt.scatter(gsdata[:, 0], gsdata[:, 1], c='blue', s=5, label='Data')
# 设置 x 和 y 轴范围
plt.xlim(gsdata[:, 0].min() , gsdata[:, 0].max() )
plt.ylim(gsdata[:, 1].min() , gsdata[:, 1].max() )
# 分别计算每个高斯分布的概率密度并绘制
colors = ['red', 'green']  # 不同高斯的颜色
density_threshold = 0.01  # 概率密度的过滤阈值

for i in range(gmm.n_components):
    mean = means[i]
    cov = covariances[i]
    rv = multivariate_normal(mean, cov)
    pdf = rv.pdf(xy).reshape(X.shape)
    
    # 过滤等高线：仅绘制概率密度大于阈值的等高线
    levels = np.linspace(density_threshold, pdf.max(), 10)  # 等高线只包含密度大于阈值的部分
    plt.contour(X, Y, pdf, levels=levels, colors=colors[i], alpha=0.5, label=f'Gaussian {i+1}')

plt.legend()
plt.show()
labels = gmm.predict(gsdata)
max_point = np.argmax(Hadamard)
core_label = labels[max_point]
core_index = np.where(labels == core_label)

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal, chi2

# 假设 gsdata 是你的数据，gmm 是已经拟合的 GMM 模型

# 提取 GMM 的参数
means = gmm.means_
covariances = gmm.covariances_
indices_in_3sigma = []
# 计算二维数据下卡方分布的 1σ 临界值
chi2_threshold_1sigma = chi2.ppf(0.6827, df=2)
# 遍历每个高斯分布
for i in range(gmm.n_components):
    mean = means[i]
    cov = covariances[i]

    # 计算数据点到均值的马氏距离
    diff = gsdata - mean
    inv_cov = np.linalg.inv(cov)
    mahalanobis_dist = np.sum(diff @ inv_cov * diff, axis=1)

    # 找到马氏距离在 3σ 范围内的点的索引
    indices = np.where(mahalanobis_dist <= chi2_threshold_1sigma)[0]
    indices_in_3sigma.append(indices)

# 输出每个高斯分布的 3σ 范围内的点的索引
for i, indices in enumerate(indices_in_3sigma):
    print(f"Gaussian {i+1} 范围内的点的索引: {indices}")


In [ ]:
common_elements = np.intersect1d(core_index, indices_in_3sigma[1])  # 手动按钮
labels = np.zeros(len(data))
labels[common_elements] = 1
coords = gsdata[common_elements]
# 计算每个点的向量长度
distances = np.linalg.norm(coords, axis=1)

# 找到最小长度对应的点
min_point = coords[np.argmin(distances)]

indices = np.where((gsdata[:, 0] > min_point[0]) & (gsdata[:, 1] > min_point[1]))[0]
ultimate = np.union1d(indices, common_elements)
labels[ultimate] = 1

In [ ]:
edges = []
to_not_sign = np.where(labels != 1)
# 试试使用Hadamard乘机作为便利的顺序
import queue

fathers = np.full((len(data), 1), np.nan)  # 记录每个点的父亲节点
is_scaned = np.zeros(len(data))  # 纪录每个点是否被扫描到
is_scaned[to_not_sign] = 1
fk_rs = np.zeros(len(data))  # 记录每个点向下扫描的半径
now_flag = 1
flags = np.zeros(len(data))  # 记录簇号
exp = []  # 记录每个簇的祖先的半径
cluster_num = []  # 记录点的数量
cluster_hearts = {}
# 记录遍历的顺序
record_order = []
while True:
    #找到每次扫描的初始点
    for i in np.argsort(Hadamard)[::-1]:
        if is_scaned[i] == 0:
            cluster_hearts[now_flag] = i
            max_point = i
            is_scaned[i] = 1
            break
        if i == np.argsort(Hadamard)[0]:
            raise Exception("所有点都被遍历了!")

    scan_queue = queue.Queue()
    scan_queue.put(max_point)
    is_scaned[max_point] = 1
    fathers[max_point] = max_point
    fk_rs[max_point] = r_s[max_point]
    exp.append(r_s[max_point])
    flags[max_point] = now_flag
    trace_dist = {}
    while not scan_queue.empty():
        item = scan_queue.get()
        record_order.append(item)
        # 查询祖先
        father_chain = []
        itself = item
        father_chain.append(itself)
        while fathers[itself] != itself:
            father_chain.append(int(fathers[itself]))
            itself = int(fathers[itself])
        # r = r_s[father_chain[-1]]
        ancestor_num = len(father_chain)
        # r = sum([r_s[e] / ancestor_num for e in father_chain])
        r = r_s[item]
        # r = 10
        fk_rs[item] = r
        indices = tree.query_ball_point(data[item], r)
        print(f'找到了{len(indices)}个')
        trace_dist[item] = indices[:]
        for e in indices[:]:
            if flags[e] and flags[e] != flags[item]:
                edges.append((flags[e], flags[item]))

            if is_scaned[e] == 0:
                scan_queue.put(e)
                fathers[e] = item
                is_scaned[e] = 1
                flags[e] = now_flag

    now_flag += 1

    # plt.scatter(x_values, y_values,marker='o',c=flags ,cmap='turbo')

In [ ]:
import networkx as nx

G = nx.Graph()
G.add_edges_from(edges)

# 找到所有连通分量
connected_components = list(nx.connected_components(G))
print("连通分量:", connected_components)
for cp in connected_components:
    cp = list(cp)
    for f in cp[1:]:
        to_be_merge = np.where(flags == f)
        flags[to_be_merge] = cp[0]
plt.scatter(x_values, y_values, c=flags, cmap='turbo', s=5)

In [ ]:
some_p = np.where(labels != 1)
is_scaned[some_p] = 0

In [ ]:
# 分配剩余的点
# 将非核心点分配到最近的簇a
for i in np.argsort(Hadamard)[::-1]:
    if is_scaned[i] == 0:
        distances, indices = tree.query(data[i], k=10)
        indices = indices[1:]
        for p in indices:
            if flags[p]:  # 如果flags 不为0 ,就把当前的点分配给这个flags
                flags[i] = flags[p]
                is_scaned[i] = 1
                break

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np


def map_array_to_colors(arr):
    """
    将输入的数组元素根据turbo颜色映射映射为对应的颜色。

    参数:
    arr (numpy.ndarray): 输入的数值数组，数组中的每个元素将被映射为一种颜色

    返回:
    list: 包含对应颜色的RGB元组的列表，每个元组对应输入数组中的一个元素的颜色
    """
    norm = mcolors.Normalize(vmin=np.min(arr), vmax=np.max(arr))
    mapper = plt.cm.ScalarMappable(norm=norm, cmap='turbo')
    return [mapper.to_rgba(x)[:3] for x in arr]
flags = np.array([int(e) for e in flags])
colors = map_array_to_colors(flags)
plt.scatter(x_values, y_values, c=colors, s=5)